# T1.4 / T1.5 — Simulador temporal e cenários de drift

Este notebook mostra como o fluxo de transações é dividido em janelas e como
os cenários `none`, `low`, `moderate` e `high` alteram a distribuição e o
conceito de fraude.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from data.drift_simulator import DRIFT_LEVELS, apply_drift
from data.loader import load_paysim
from data.temporal_simulator import TemporalSimulator

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
pd.DataFrame(DRIFT_LEVELS).T

## 1. Janelas temporais

As janelas são contíguas e disjuntas; nenhuma transação aparece em duas.

In [ ]:
df = load_paysim(rows=40_000, seed=42)
simulator = TemporalSimulator(df, n_windows=12, seed=42)
windows = simulator.windows()
pd.DataFrame([window.summary() for window in windows])

## 2. Efeito do drift nos valores

O simulador combina **covariate shift** (valores/saldos e mix de tipos) com
**concept drift** (nova tipologia de fraude que preserva saldo na origem).

In [ ]:
sample = df.sample(15_000, random_state=42).reset_index(drop=True)
rows = []
for level in DRIFT_LEVELS:
    drifted = apply_drift(sample, level, seed=42)
    rows.append(
        {
            "cenario": level,
            "amount_medio": drifted["amount"].mean(),
            "fracao_fraude": drifted["is_fraud"].mean(),
            "fracao_transfer": (drifted["type"] == "TRANSFER").mean(),
        }
    )
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for level, color in zip(DRIFT_LEVELS, ["#64748b", "#0ea5e9", "#f59e0b", "#dc2626"]):
    values = apply_drift(sample, level, seed=42)["amount"]
    sns.kdeplot(values.clip(upper=values.quantile(0.99)), label=level, color=color, ax=axes[0])
axes[0].set_title("Distribuição de amount por cenário")
axes[0].set_xlabel("valor")

means = [apply_drift(sample, level, seed=42)["amount"].mean() for level in DRIFT_LEVELS]
axes[1].bar(list(DRIFT_LEVELS), means, color=["#64748b", "#0ea5e9", "#f59e0b", "#dc2626"])
axes[1].set_title("Valor médio por cenário")
axes[1].set_ylabel("amount médio")

plt.tight_layout()
plt.show()

## 3. Nova tipologia de fraude

No cenário forte parte das fraudes usa uma conta de destino recém-criada e
**preserva saldo na origem** — combinação ausente no treino do baseline.

In [ ]:
base = sample[sample["is_fraud"] & (sample["new_balance_org"] <= 0)].head(3)
base[["type", "amount", "old_balance_org", "new_balance_org", "old_balance_dest", "new_balance_dest"]]

In [ ]:
drifted = apply_drift(sample, "high", seed=42)
novo_padrao = drifted[
    drifted["is_fraud"]
    & (drifted["new_balance_org"] > 0)
    & (drifted["old_balance_dest"] == 0)
]
print(f"Fraudes da nova tipologia: {len(novo_padrao)}")
novo_padrao.head(3)[["type", "amount", "old_balance_org", "new_balance_org", "old_balance_dest", "new_balance_dest"]]

## 4. Ramp de drift (usado no E4)

A intensidade cresce ao longo das janelas, simulando a passagem do tempo até
o cenário forte.

In [ ]:
ramp = simulator.generate(level="high", ramp=True, ramp_windows=5)
pd.DataFrame(
    [
        {"janela": window.label, "intensidade": window.drift_intensity, "linhas": window.rows}
        for window in ramp
    ]
)